In [40]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print("Imports successful")

Imports successful


In [41]:
project_root = Path.cwd().parent

data_path = project_root / "data" / "processed" / "clickbait.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Dataset loaded successfully
Shape: (19538, 4)
Columns: ['id', 'headline', 'clickbait_score', 'clickbait_label']


In [42]:
X = df["headline"]
y = df["clickbait_label"]

print("Features:", X.shape)
print("Target:", y.shape)

print("\nClasses:")
print(y.value_counts())

Features: (19538,)
Target: (19538,)

Classes:
clickbait_label
no-clickbait    14777
clickbait        4761
Name: count, dtype: int64


In [43]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

Training: 15630
Validation: 1954
Test: 1954


In [44]:
print("TRAIN")
print(y_train.value_counts(normalize=True))

print("\nVALIDATION")
print(y_val.value_counts(normalize=True))

print("\nTEST")
print(y_test.value_counts(normalize=True))

TRAIN
clickbait_label
no-clickbait    0.756302
clickbait       0.243698
Name: proportion, dtype: float64

VALIDATION
clickbait_label
no-clickbait    0.756397
clickbait       0.243603
Name: proportion, dtype: float64

TEST
clickbait_label
no-clickbait    0.756397
clickbait       0.243603
Name: proportion, dtype: float64


In [45]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)

X_val_tfidf = tfidf.transform(X_val)

X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (15630, 29147)
Validation TF-IDF shape: (1954, 29147)
Test TF-IDF shape: (1954, 29147)


In [46]:
print("Number of TF-IDF features:", len(tfidf.get_feature_names_out()))

Number of TF-IDF features: 29147


In [47]:
clickbait_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

clickbait_model.fit(X_train_tfidf, y_train)

print("Clickbait model trained successfully")

Clickbait model trained successfully


In [48]:
y_val_pred = clickbait_model.predict(X_val_tfidf)

print(classification_report(y_val, y_val_pred))

              precision    recall  f1-score   support

   clickbait       0.63      0.25      0.36       476
no-clickbait       0.80      0.95      0.87      1478

    accuracy                           0.78      1954
   macro avg       0.72      0.60      0.62      1954
weighted avg       0.76      0.78      0.75      1954



In [49]:
y_val_proba = clickbait_model.predict_proba(X_val_tfidf)

clickbait_index = list(clickbait_model.classes_).index("clickbait")

clickbait_probabilities = y_val_proba[:, clickbait_index]

roc_auc = roc_auc_score(
    (y_val == "clickbait").astype(int),
    clickbait_probabilities
)

print("Validation ROC-AUC:", roc_auc)

Validation ROC-AUC: 0.7549791905936936


In [50]:
cm = confusion_matrix(
    y_val,
    y_val_pred,
    labels=["no-clickbait", "clickbait"]
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[1408   70]
 [ 355  121]]


In [51]:
balanced_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

balanced_model.fit(X_train_tfidf, y_train)

print("Balanced Logistic Regression trained successfully")

Balanced Logistic Regression trained successfully


In [52]:
y_val_balanced_pred = balanced_model.predict(X_val_tfidf)

print(classification_report(y_val, y_val_balanced_pred))

              precision    recall  f1-score   support

   clickbait       0.49      0.60      0.54       476
no-clickbait       0.86      0.80      0.83      1478

    accuracy                           0.75      1954
   macro avg       0.67      0.70      0.68      1954
weighted avg       0.77      0.75      0.76      1954



In [53]:
y_val_balanced_proba = balanced_model.predict_proba(X_val_tfidf)

balanced_clickbait_index = list(
    balanced_model.classes_
).index("clickbait")

balanced_clickbait_probabilities = (
    y_val_balanced_proba[:, balanced_clickbait_index]
)

balanced_roc_auc = roc_auc_score(
    (y_val == "clickbait").astype(int),
    balanced_clickbait_probabilities
)

print("Balanced Logistic Regression ROC-AUC:", balanced_roc_auc)

Balanced Logistic Regression ROC-AUC: 0.7551170671245494


In [54]:
balanced_cm = confusion_matrix(
    y_val,
    y_val_balanced_pred,
    labels=["no-clickbait", "clickbait"]
)

print("Balanced Confusion Matrix:")
print(balanced_cm)

Balanced Confusion Matrix:
[[1177  301]
 [ 192  284]]


In [55]:
from sklearn.metrics import f1_score, precision_score, recall_score

threshold_results = []

for threshold in np.arange(0.20, 0.81, 0.05):
    predictions = np.where(
        balanced_clickbait_probabilities >= threshold,
        "clickbait",
        "no-clickbait"
    )

    precision = precision_score(
        y_val,
        predictions,
        pos_label="clickbait"
    )

    recall = recall_score(
        y_val,
        predictions,
        pos_label="clickbait"
    )

    f1 = f1_score(
        y_val,
        predictions,
        pos_label="clickbait"
    )

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df)

,threshold,precision,recall,f1
0,0.20,0.269299,0.960084,0.420617
1,0.25,0.298780,0.926471,0.451844
2,0.30,0.327711,0.857143,0.474143
3,0.35,0.361785,0.783613,0.495023
4,0.40,0.401869,0.722689,0.516517
5,0.45,0.441429,0.649160,0.525510
6,0.50,0.485470,0.596639,0.535344
7,0.55,0.515400,0.527311,0.521288
8,0.60,0.555270,0.453782,0.499422
9,0.65,0.601266,0.399160,0.479798


In [56]:
best_threshold_row = threshold_df.loc[
    threshold_df["f1"].idxmax()
]

print("Best threshold:")
print(best_threshold_row)

Best threshold:
threshold    0.500000
precision    0.485470
recall       0.596639
f1           0.535344
Name: 6, dtype: float64


In [57]:
from sklearn.svm import LinearSVC

print("LinearSVC imported successfully")

LinearSVC imported successfully


In [58]:
svm_model = LinearSVC(
    class_weight="balanced",
    random_state=42,
    max_iter=5000
)

svm_model.fit(X_train_tfidf, y_train)

print("Linear SVM trained successfully")

Linear SVM trained successfully


In [59]:
y_val_svm_pred = svm_model.predict(X_val_tfidf)

print(classification_report(y_val, y_val_svm_pred))

              precision    recall  f1-score   support

   clickbait       0.48      0.52      0.50       476
no-clickbait       0.84      0.82      0.83      1478

    accuracy                           0.75      1954
   macro avg       0.66      0.67      0.67      1954
weighted avg       0.75      0.75      0.75      1954



In [60]:
svm_cm = confusion_matrix(
    y_val,
    y_val_svm_pred,
    labels=["no-clickbait", "clickbait"]
)

print("Linear SVM Confusion Matrix:")
print(svm_cm)

Linear SVM Confusion Matrix:
[[1212  266]
 [ 227  249]]


In [61]:
svm_scores = svm_model.decision_function(X_val_tfidf)

In [62]:
svm_roc_auc = roc_auc_score(
    (y_val == "clickbait").astype(int),
    svm_scores
)
print("Linear SVM ROC-AUC:", svm_roc_auc)

Linear SVM ROC-AUC: 0.26904970377866977


In [63]:
print("SVM classes:", svm_model.classes_)

SVM classes: ['clickbait' 'no-clickbait']


In [64]:
svm_scores = svm_model.decision_function(X_val_tfidf)

svm_roc_auc_corrected = roc_auc_score(
    (y_val == "clickbait").astype(int),
    -svm_scores
)

print("Corrected Linear SVM ROC-AUC:", svm_roc_auc_corrected)

Corrected Linear SVM ROC-AUC: 0.7309502962213302


In [65]:
import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix

curiosity_phrases = [
    "won't believe",
    "wont believe",
    "this is why",
    "here's why",
    "heres why",
    "what happens next",
    "will blow your mind",
    "you need to know",
    "will make you",
    "doesn't want you to know",
    "doesnt want you to know",
    "unbelievable",
    "shocking",
    "secret"
]

def extract_handcrafted_features(texts):
    features = []

    for text in texts:
        text = str(text)
        text_lower = text.lower()

        alpha_chars = [
            c for c in text
            if c.isalpha()
        ]

        uppercase_ratio = (
            sum(c.isupper() for c in alpha_chars)
            / len(alpha_chars)
            if alpha_chars
            else 0.0
        )
        features.append({
            "exclamation_count": text.count("!"),
            "question_count": text.count("?"),
            "you_count": len(
                re.findall(r"\byou\b", text_lower)
            ),
            "your_count": len(
                re.findall(r"\byour\b", text_lower)
            ),
            "starts_with_number": int(
                bool(re.match(r"^\s*\d", text))
            ),
            "uppercase_ratio": uppercase_ratio,
            "word_count": len(text.split()),
            "char_count": len(text),
            "curiosity_phrase_count": sum(
                text_lower.count(p)
                for p in curiosity_phrases
            ),
            "has_superlative": int(
                bool(
                    re.search(
                        r"\b(best|worst|most|ever|ultimate|"
                        r"biggest|craziest)\b",
                        text_lower
                    )
                )
            )
        })
    return pd.DataFrame(features)
train_handcrafted = extract_handcrafted_features(
    X_train
).reset_index(drop=True)

val_handcrafted = extract_handcrafted_features(
    X_val
).reset_index(drop=True)

test_handcrafted = extract_handcrafted_features(
    X_test
).reset_index(drop=True)

assert len(train_handcrafted) == X_train_tfidf.shape[0]
assert len(val_handcrafted) == X_val_tfidf.shape[0]
assert len(test_handcrafted) == X_test_tfidf.shape[0]

print("Row alignment check: PASSED")
print("\nHandcrafted feature shape:")
print("Train:", train_handcrafted.shape)
print("Validation:", val_handcrafted.shape)
print("Test:", test_handcrafted.shape)
display(train_handcrafted.head())
feature_scaler = StandardScaler()
train_handcrafted_scaled = feature_scaler.fit_transform(
    train_handcrafted
)
val_handcrafted_scaled = feature_scaler.transform(
    val_handcrafted
)
test_handcrafted_scaled = feature_scaler.transform(
    test_handcrafted
)
X_train_combined = hstack([
    X_train_tfidf,
    csr_matrix(train_handcrafted_scaled)
])
X_val_combined = hstack([
    X_val_tfidf,
    csr_matrix(val_handcrafted_scaled)
])
X_test_combined = hstack([
    X_test_tfidf,
    csr_matrix(test_handcrafted_scaled)
])
print("\nCombined feature shapes:")
print("Training:", X_train_combined.shape)
print("Validation:", X_val_combined.shape)
print("Test:", X_test_combined.shape)
assert X_train_combined.shape[1] == X_train_tfidf.shape[1] + 10
assert X_val_combined.shape[1] == X_val_tfidf.shape[1] + 10
assert X_test_combined.shape[1] == X_test_tfidf.shape[1] + 10
print("\nTF-IDF + handcrafted feature engineering: PASSED")

Row alignment check: PASSED

Handcrafted feature shape:
Train: (15630, 10)
Validation: (1954, 10)
Test: (1954, 10)


,exclamation_count,question_count,you_count,your_count,starts_with_number,uppercase_ratio,word_count,char_count,curiosity_phrase_count,has_superlative
0,0,1,0,0,0,0.084746,10,75,0,0
1,0,0,0,0,0,0.027027,10,52,0,0
2,0,0,0,0,0,0.140625,11,76,0,0
3,0,0,0,0,0,0.014085,16,86,0,0
4,0,0,0,0,0,0.147059,9,43,0,0



Combined feature shapes:
Training: (15630, 29157)
Validation: (1954, 29157)
Test: (1954, 29157)

TF-IDF + handcrafted feature engineering: PASSED


In [66]:
combined_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)
combined_model.fit( X_train_combined, y_train)
print("TF-IDF + handcrafted Logistic Regression trained successfully")

TF-IDF + handcrafted Logistic Regression trained successfully


In [67]:
# Predict on validation set
y_val_combined_pred = combined_model.predict(
    X_val_combined
)
# Classification report
print("TF-IDF + Handcrafted Features")
print("=" * 50)
print(classification_report(
    y_val,
    y_val_combined_pred
))

TF-IDF + Handcrafted Features
              precision    recall  f1-score   support

   clickbait       0.50      0.60      0.54       476
no-clickbait       0.86      0.80      0.83      1478

    accuracy                           0.75      1954
   macro avg       0.68      0.70      0.69      1954
weighted avg       0.77      0.75      0.76      1954



In [68]:
# Get clickbait probabilities
combined_proba = combined_model.predict_proba(X_val_combined)
combined_clickbait_index = list(combined_model.classes_).index("clickbait")
combined_clickbait_probabilities = (combined_proba[:, combined_clickbait_index])
combined_roc_auc = roc_auc_score(
    (y_val == "clickbait").astype(int),combined_clickbait_probabilities)
print("TF-IDF + Handcrafted ROC-AUC:", combined_roc_auc)

TF-IDF + Handcrafted ROC-AUC: 0.76409325570553


In [69]:
combined_cm = confusion_matrix(
    y_val,
    y_val_combined_pred,
    labels=["no-clickbait", "clickbait"]
)
print("TF-IDF + Handcrafted Confusion Matrix:")
print(combined_cm)

TF-IDF + Handcrafted Confusion Matrix:
[[1189  289]
 [ 191  285]]


In [70]:
threshold_results_combined = []

for threshold in np.arange(0.20, 0.81, 0.05):

    predictions = np.where(
        combined_clickbait_probabilities >= threshold,
        "clickbait",
        "no-clickbait"
    )

    precision = precision_score(
        y_val,
        predictions,
        pos_label="clickbait"
    )

    recall = recall_score(
        y_val,
        predictions,
        pos_label="clickbait"
    )

    f1 = f1_score(
        y_val,
        predictions,
        pos_label="clickbait"
    )

    threshold_results_combined.append({
        "threshold": round(threshold, 2),
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

combined_threshold_df = pd.DataFrame(
    threshold_results_combined
)

display(combined_threshold_df)

,threshold,precision,recall,f1
0,0.20,0.279169,0.960084,0.432560
1,0.25,0.309950,0.909664,0.462360
2,0.30,0.336465,0.827731,0.478446
3,0.35,0.377397,0.785714,0.509884
4,0.40,0.411408,0.712185,0.521538
5,0.45,0.453890,0.661765,0.538462
6,0.50,0.496516,0.598739,0.542857
7,0.55,0.528184,0.531513,0.529843
8,0.60,0.551637,0.460084,0.501718
9,0.65,0.562500,0.397059,0.465517


In [71]:
best_combined_threshold = combined_threshold_df.loc[
    combined_threshold_df["f1"].idxmax()
]
print("Best threshold for combined model:")
print(best_combined_threshold)

Best threshold for combined model:
threshold    0.500000
precision    0.496516
recall       0.598739
f1           0.542857
Name: 6, dtype: float64


In [72]:
# FINAL TEST EVALUATION: CLICKBAIT MODEL
# TF-IDF + HANDCRAFTED FEATURES
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
# 1. Predict labels on the held-out test set
y_test_combined_pred = combined_model.predict(
    X_test_combined
)
# 2. Obtain clickbait probabilities
y_test_combined_proba = combined_model.predict_proba(
    X_test_combined
)
clickbait_index = list(
    combined_model.classes_
).index("clickbait")
y_test_clickbait_prob = y_test_combined_proba[
    :, clickbait_index
]
# 3. Evaluate the model
print("========== FINAL CLICKBAIT TEST RESULTS ==========")
print(
    "Accuracy:",
    round(
        accuracy_score(y_test, y_test_combined_pred),
        4
    )
)
print("ROC-AUC:", round( roc_auc_score((y_test == "clickbait").astype(int), y_test_clickbait_prob),4))
print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_test_combined_pred,
        labels=["clickbait", "no-clickbait"],
        digits=4
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_test_combined_pred,
        labels=["no-clickbait", "clickbait"]
    )
)

========== FINAL CLICKBAIT TEST RESULTS ==========
Accuracy: 0.762
ROC-AUC: 0.7771

Classification Report:
              precision    recall  f1-score   support

   clickbait     0.5098    0.5987    0.5507       476
no-clickbait     0.8631    0.8146    0.8381      1478

    accuracy                         0.7620      1954
   macro avg     0.6865    0.7067    0.6944      1954
weighted avg     0.7770    0.7620    0.7681      1954

Confusion Matrix:
[[1204  274]
 [ 191  285]]


In [73]:
# SAVE COMPLETE CLICKBAIT MODEL ARTIFACTS
import joblib
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Confirm required fitted objects exist
required_objects = [
    "combined_model",
    "tfidf",
    "feature_scaler",
]
missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Missing required objects: {missing_objects}"
    )

# Save the fitted components separately
joblib.dump(
    combined_model,
    MODEL_DIR / "clickbait_model.joblib"
)

joblib.dump(
    tfidf,
    MODEL_DIR / "clickbait_tfidf.joblib"
)

joblib.dump(
    feature_scaler,
    MODEL_DIR / "clickbait_scaler.joblib"
)

# Verify saved files
for filename in [
    "clickbait_model.joblib",
    "clickbait_tfidf.joblib",
    "clickbait_scaler.joblib",
]:
    path = MODEL_DIR / filename
    assert path.exists(), f"Missing file: {path}"
    print(f"Saved and verified: {path}")
print("\nClickbait model components saved.")

Saved and verified: c:\Users\KIIT\clickbait-credibility-system\models\clickbait_model.joblib
Saved and verified: c:\Users\KIIT\clickbait-credibility-system\models\clickbait_tfidf.joblib
Saved and verified: c:\Users\KIIT\clickbait-credibility-system\models\clickbait_scaler.joblib

Clickbait model components saved.


In [75]:
import joblib

saved_model = joblib.load(MODEL_DIR / "clickbait_model.joblib")
saved_tfidf = joblib.load(MODEL_DIR / "clickbait_tfidf.joblib")
saved_scaler = joblib.load(MODEL_DIR / "clickbait_scaler.joblib")

assert type(saved_model) is type(combined_model)
assert type(saved_tfidf) is type(tfidf)
assert type(saved_scaler) is type(feature_scaler)

assert saved_model.n_features_in_ == (
    len(saved_tfidf.get_feature_names_out()) + 10
)

print("Artifact verification passed.")
print("Classifier input features:", saved_model.n_features_in_)
print("TF-IDF features:", len(saved_tfidf.get_feature_names_out()))
print("Handcrafted features:", 10)

Artifact verification passed.
Classifier input features: 29157
TF-IDF features: 29147
Handcrafted features: 10


In [76]:
# STEP 6: TEST CLICKBAIT MODEL ON NEW HEADLINES
import re
import numpy as np
import pandas as pd
import joblib
from scipy.sparse import hstack, csr_matrix
from pathlib import Path
# Load the saved artifacts
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

model_dir = project_root / "models"
loaded_model = joblib.load(model_dir / "clickbait_model.joblib")
loaded_tfidf = joblib.load(model_dir / "clickbait_tfidf.joblib")
loaded_scaler = joblib.load(model_dir / "clickbait_scaler.joblib")

print("Saved clickbait artifacts loaded successfully.")
# Extract the same 10 handcrafted features used during training
def extract_features_for_prediction(texts):
    curiosity_phrases = [
        "won't believe",
        "wont believe",
        "this is why",
        "here's why",
        "heres why",
        "what happens next",
        "will blow your mind",
        "you need to know",
        "will make you",
        "doesn't want you to know",
        "doesnt want you to know",
        "unbelievable",
        "shocking",
        "secret"
    ]

    features = []

    for text in texts:
        text = str(text)
        text_lower = text.lower()

        alpha_chars = [c for c in text if c.isalpha()]

        uppercase_ratio = (
            sum(c.isupper() for c in alpha_chars) / len(alpha_chars)
            if alpha_chars else 0.0
        )

        features.append({
            "exclamation_count": text.count("!"),
            "question_count": text.count("?"),
            "you_count": len(re.findall(r"\byou\b", text_lower)),
            "your_count": len(re.findall(r"\byour\b", text_lower)),
            "starts_with_number": int(
                bool(re.match(r"^\s*\d", text))
            ),
            "uppercase_ratio": uppercase_ratio,
            "word_count": len(text.split()),
            "char_count": len(text),
            "curiosity_phrase_count": sum(
                text_lower.count(phrase)
                for phrase in curiosity_phrases
            ),
            "has_superlative": int(
                bool(re.search(
                    r"\b(best|worst|most|ever|ultimate|"
                    r"biggest|craziest)\b",
                    text_lower
                ))
            )
        })

    return pd.DataFrame(features)


# Predict clickbait for new headlines
def predict_clickbait(headlines):
    headlines = [str(headline) for headline in headlines]

    # TF-IDF features
    tfidf_features = loaded_tfidf.transform(headlines)

    # Handcrafted features in the same order used during training
    handcrafted_df = extract_features_for_prediction(headlines)

    handcrafted_features = loaded_scaler.transform(handcrafted_df)

    # Combine both feature groups
    combined_features = hstack([
        tfidf_features,
        csr_matrix(handcrafted_features)
    ]).tocsr()

    # Confirm the model receives the expected feature count
    assert combined_features.shape[1] == loaded_model.n_features_in_, (
        f"Feature mismatch: received {combined_features.shape[1]}, "
        f"expected {loaded_model.n_features_in_}"
    )

    # Probability of the clickbait class
    classes = list(loaded_model.classes_)
    clickbait_index = classes.index("clickbait")

    probabilities = loaded_model.predict_proba(
        combined_features
    )[:, clickbait_index]

    results = []

    for headline, probability in zip(headlines, probabilities):
        results.append({
            "headline": headline,
            "clickbait_probability": round(float(probability), 4),
            "predicted_label": (
                "clickbait" if probability >= 0.5
                else "no-clickbait"
            )
        })
    return pd.DataFrame(results)
# Try a few example headlines
sample_headlines = [
    "You Won't Believe What Happened Next!",
    "Government Announces New Economic Policy",
    "The Secret Trick That Will Change Your Life",
    "Local Schools Announce Revised Examination Schedule"
]
display(predict_clickbait(sample_headlines))

Saved clickbait artifacts loaded successfully.


,headline,clickbait_probability,predicted_label
0,You Won't Believe What Happened Next!,0.9105,clickbait
1,Government Announces New Economic Policy,0.2642,no-clickbait
2,The Secret Trick That Will Change Your Life,0.8919,clickbait
3,Local Schools Announce Revised Examination Sch...,0.1483,no-clickbait


In [78]:
# STEP 6A: VERIFY CLICKBAIT MODEL CONFIGURATION

print("Model type:", type(loaded_model).__name__)
print("TF-IDF vocabulary size:", len(loaded_tfidf.vocabulary_))
print("Scaler feature count:", loaded_scaler.n_features_in_)
print("Model input feature count:", loaded_model.n_features_in_)

print("\nModel classes:", loaded_model.classes_)

print("\nScaler feature names:")
print(list(loaded_scaler.feature_names_in_))

print("\nTF-IDF + handcrafted feature count:")
print(
    len(loaded_tfidf.vocabulary_)
    + loaded_scaler.n_features_in_
)

Model type: LogisticRegression
TF-IDF vocabulary size: 29147
Scaler feature count: 10
Model input feature count: 29157

Model classes: ['clickbait' 'no-clickbait']

Scaler feature names:
['exclamation_count', 'question_count', 'you_count', 'your_count', 'starts_with_number', 'uppercase_ratio', 'word_count', 'char_count', 'curiosity_phrase_count', 'has_superlative']

TF-IDF + handcrafted feature count:
29157


In [79]:
# STEP 6A: VERIFY SAVED CLICKBAIT ARTIFACTS
print("Model type:", type(loaded_model).__name__)
print("TF-IDF vocabulary size:", len(loaded_tfidf.vocabulary_))
print("Scaler feature count:", loaded_scaler.n_features_in_)
print("Model input feature count:", loaded_model.n_features_in_)
print("\nModel classes:")
print(loaded_model.classes_)
print("\nExpected total feature count:")
print(
    len(loaded_tfidf.vocabulary_)
    + loaded_scaler.n_features_in_
)
print("\nActual model input feature count:")
print(loaded_model.n_features_in_)
assert (
    len(loaded_tfidf.vocabulary_)
    + loaded_scaler.n_features_in_
    == loaded_model.n_features_in_
), "ERROR: Saved model and preprocessing artifacts do not match."
print("\nArtifact dimension check: PASSED")

Model type: LogisticRegression
TF-IDF vocabulary size: 29147
Scaler feature count: 10
Model input feature count: 29157

Model classes:
['clickbait' 'no-clickbait']

Expected total feature count:
29157

Actual model input feature count:
29157

Artifact dimension check: PASSED


In [80]:
# STEP 6B: VERIFY HANDCRAFTED FEATURE ORDER
expected_feature_names = [
    "exclamation_count",
    "question_count",
    "you_count",
    "your_count",
    "starts_with_number",
    "uppercase_ratio",
    "word_count",
    "char_count",
    "curiosity_phrase_count",
    "has_superlative"
]
sample_features = extract_features_for_prediction([
    "You Won't Believe What Happened Next!"
])
print("Prediction feature columns:")
print(sample_features.columns.tolist())
assert sample_features.columns.tolist() == expected_feature_names, (
    "ERROR: Handcrafted feature names or their order do not match training."
)
print("\nHandcrafted feature-order check: PASSED")

Prediction feature columns:
['exclamation_count', 'question_count', 'you_count', 'your_count', 'starts_with_number', 'uppercase_ratio', 'word_count', 'char_count', 'curiosity_phrase_count', 'has_superlative']

Handcrafted feature-order check: PASSED


In [81]:
# STEP 6C: TEST CLICKBAIT PREDICTIONS
sample_headlines = [
    "You Won't Believe What Happened Next!",
    "Government Announces New Economic Policy",
    "The Secret Trick That Will Change Your Life",
    "Local Schools Announce Revised Examination Schedule"
]
results = predict_clickbait(sample_headlines)
display(results)

,headline,clickbait_probability,predicted_label
0,You Won't Believe What Happened Next!,0.9105,clickbait
1,Government Announces New Economic Policy,0.2642,no-clickbait
2,The Secret Trick That Will Change Your Life,0.8919,clickbait
3,Local Schools Announce Revised Examination Sch...,0.1483,no-clickbait
